# Modelo de Ising — Algoritmo de Metropolis-Hastings

Implementação da classe `IsingModel` com condições de fronteira periódicas,  
`J = 1`, `h = 0`, `kb = 1`, rede 2D quadrada de lado `L`, spins iniciais `+1`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import curve_fit
from scipy.integrate import cumulative_trapezoid
from scipy.interpolate import CubicSpline

J  = 1    # constante de acoplamento
h  = 0    # CORRIGIDO: era h=0.1 — o enunciado especifica h=0
kb = 1    # constante de Boltzmann
rnd = np.random.default_rng(seed=1)  # gerador de números aleatórios


# ── Classe do Modelo de Ising ────────────────────────────────────────────────
class IsingModel:

    def __init__(self, L, T):
        # parâmetros do modelo
        self.L = L
        self.T = T

        # rede com todos os spins = +1
        self.spins = np.ones((L, L))

        # listas para guardar a evolução das variáveis
        self.energies       = []
        self.magnetizations = []

    # ── energia de um spin individual (conta apenas vizinhos à direita e abaixo
    #    para evitar dupla contagem quando somado sobre todos os spins) ─────────
    def calc_ener_spin(self, i, j) -> float:
        L    = self.L
        own  = self.spins[i, j]
        # CORRIGIDO: substituído método neighbours (usava np.isin de forma errada)
        #            por acesso directo com índices periódicos via %
        right = self.spins[i,         (j + 1) % L]
        down  = self.spins[(i + 1) % L,  j       ]
        energy = -J * own * (right + down) - h * own
        return energy

    # ── energia total do sistema ─────────────────────────────────────────────
    def calc_ener(self) -> float:
        # CORRIGIDO: substituído duplo loop Python por operações numpy vectorizadas
        #            (ordens de grandeza mais rápido para L grandes)
        right  = np.roll(self.spins, -1, axis=1)   # vizinhos à direita
        down   = np.roll(self.spins, -1, axis=0)   # vizinhos de baixo
        energy = -J * np.sum(self.spins * (right + down)) - h * np.sum(self.spins)
        return energy

    # ── magnetização por spin m = M/L² ───────────────────────────────────────
    def calc_mag(self) -> float:
        M   = J * np.sum(self.spins)
        mag = M / self.L**2
        return mag

    # ── inverte o spin (i,j) ─────────────────────────────────────────────────
    def flip(self, i, j) -> None:
        self.spins[i, j] *= -1

    # ── iteração de Monte Carlo (Metropolis-Hastings) ─────────────────────────
    def iter_monte_carlo(self, n_iter):
        L = self.L

        # CORRIGIDO: arrays numpy pré-alocados em vez de listas com append
        #            (evita realocação de memória repetida → mais eficiente)
        self.energies       = np.empty(n_iter + 1)
        self.magnetizations = np.empty(n_iter + 1)

        # estado inicial
        E   = self.calc_ener()
        mag = self.calc_mag()
        self.energies[0]       = E
        self.magnetizations[0] = mag

        # CORRIGIDO (optimização): pré-gerar todos os números aleatórios fora
        #           do loop reduz o overhead de chamadas repetidas ao gerador
        i_vals = rnd.integers(0, L,        size=n_iter)
        j_vals = rnd.integers(0, L,        size=n_iter)
        u_vals = rnd.uniform (0.0, 1.0,    size=n_iter)

        for k in tqdm(range(n_iter), desc=f"L={L:6d}, T={self.T:8.4f}"):
            i   = i_vals[k]
            j   = j_vals[k]
            u   = u_vals[k]
            own = self.spins[i, j]   # spin antes do flip

            # CORRIGIDO: substituído método neighbours (usava np.isin de forma errada)
            #            Acesso directo aos 4 vizinhos com índices periódicos via %
            sumviz = (self.spins[i,        (j + 1) % L] +
                      self.spins[i,        (j - 1) % L] +
                      self.spins[(i + 1) % L,  j       ] +
                      self.spins[(i - 1) % L,  j       ])

            delta = 2.0 * own * (J * sumviz + h)   # ΔE ao flippar o spin

            if delta < 0 or np.exp(-delta / (kb * self.T)) > u:
                # flip aceite: actualizar spin, energia e magnetização
                self.flip(i, j)
                E   += delta
                # CORRIGIDO: magnetização actualizada incrementalmente
                #            (era calc_mag() a cada passo → custo O(L²) por iteração)
                #            quando own flipa para -own: ΔM = J*(-2*own) → Δmag = -2*J*own/L²
                mag -= 2.0 * J * own / L**2

            self.energies[k + 1]       = E
            self.magnetizations[k + 1] = mag

    @property
    def energy(self):
        # aceder ao array com as energias
        return np.array(self.energies)

    @property
    def magnetization(self):
        # aceder ao array com as magnetizações
        return np.array(self.magnetizations)

---
## Exercício 1 — Tempo de Termalização

### 1(a)

Para cada combinação de $L \in \{16, 32, 64, 128\}$ e $T \in \{1, 2, 3, 4\}$,
corre a simulação e representa a energia média por spin

$$e \equiv \frac{E}{L^2}$$

em função do número de iteração $N$.

O número de iterações é escalado com $L^2$ (número de spins) para que o número
de *varrimentos* (sweeps $= N/L^2$) seja idêntico para todos os tamanhos.

In [ ]:
# ── Parâmetros do exercício 1(a) ─────────────────────────────────────────────
L_values = [16, 32, 64, 128]
T_values = [1, 2, 3, 4]
n_sweeps = 10    # número de varrimentos (sweeps) por simulação
                 # n_iter = n_sweeps * L² para cada L
                 # 10 sweeps é suficiente para ver o transiente;
                 # não precisamos de esperar pela termalização completa —
                 # é exactamente τterm que estamos a tentar medir

# ── Simulações ───────────────────────────────────────────────────────────────
# Guardar e = E/L² para cada (L, T)
results = {}

for L in L_values:
    n_iter = n_sweeps * L**2   # nº de iterações escalonado com L²
    for T in T_values:
        model = IsingModel(L, T)
        model.iter_monte_carlo(n_iter)
        results[(L, T)] = model.energy / L**2   # e = E/L²

# ── Gráficos — grelha 4×4: linhas = L, colunas = T ──────────────────────────
fig, axes = plt.subplots(
    len(L_values), len(T_values),
    figsize=(16, 12),
    sharex=False, sharey=False
)
fig.suptitle(
    r"Exercício 1(a) — energia por spin $e = E/L^2$ em função da iteração $N$",
    fontsize=14
)

cores = ["C0", "C1", "C2", "C3"]   # uma cor por temperatura

for row, L in enumerate(L_values):
    for col, T in enumerate(T_values):
        ax = axes[row, col]
        e  = results[(L, T)]
        N  = np.arange(len(e))         # eixo x: número de iteração

        ax.plot(N, e, lw=0.6, color=cores[col])
        ax.set_title(f"$L = {L}$,  $T = {T}$", fontsize=10)
        ax.grid(True, ls="--", alpha=0.4)

        # rótulos apenas nas bordas exteriores
        if row == len(L_values) - 1:
            ax.set_xlabel("$N$ (iteração)")
        if col == 0:
            ax.set_ylabel(r"$e = E/L^2$")

plt.tight_layout()
plt.show()

### 1(b)

Fit da energia por spin à função de termalização:

$$e(N) = e_f + (e_0 - e_f)\,e^{-N/\tau_\mathrm{term}}$$

- $e_0 = -2$ — energia inicial (todos os spins `+1`, fixo)
- $e_f(L,T)$ — energia de equilíbrio
- $\tau_\mathrm{term}(L,T)$ — tempo de termalização

Usa-se `scipy.optimize.curve_fit`. Os dados são subamostrados a ~500 pontos
para reduzir ruído antes do fit.

In [ ]:
# ── Exercício 1(b) ───────────────────────────────────────────────────────────

# Função de termalização — e0 = -2 fixo (todos os spins começam a +1)
def therm_model(N, ef, tau):
    e0 = -2.0
    return ef + (e0 - ef) * np.exp(-N / tau)

fit_params = {}   # (L, T) -> array([ef, tau])
fit_errors  = {}  # (L, T) -> array([sigma_ef, sigma_tau])

for L in L_values:
    for T in T_values:
        e = results[(L, T)]
        N = np.arange(len(e), dtype=float)

        # Subamostrar para ~500 pontos: reduz ruído MC e custo do fit
        step  = max(1, len(e) // 500)
        e_sub = e[::step]
        N_sub = N[::step]

        # Estimativas iniciais
        n_tail = max(1, len(e_sub) // 10)
        ef_p0  = float(e_sub[-n_tail:].mean())  # média do último 10% → estimativa de ef
        tau_p0 = float(L**2)                     # ordem de grandeza típica de tau

        try:
            popt, pcov = curve_fit(
                therm_model, N_sub, e_sub,
                p0    = [ef_p0, tau_p0],
                # ef ∈ [-2.5, 0.5]  (ordenado→desordenado, com margem numérica)
                # tau ∈ [1, +∞)     (deve ser positivo)
                bounds = ([-2.5, 1.0], [0.5, np.inf]),
                maxfev = 10000
            )
            perr = np.sqrt(np.diag(pcov))
        except (RuntimeError, ValueError):
            # fit não convergiu (pode acontecer a T=1 onde a curva é quase plana)
            popt = np.full(2, np.nan)
            perr = np.full(2, np.nan)

        fit_params[(L, T)] = popt
        fit_errors [(L, T)] = perr

# ── Gráficos: dados + fit ────────────────────────────────────────────────────
fig, axes = plt.subplots(len(L_values), len(T_values), figsize=(16, 12))
fig.suptitle(
    r"Exercício 1(b) — fit $e(N)=e_f+(e_0-e_f)\,e^{-N/\tau_\mathrm{term}}$",
    fontsize=13
)

for row, L in enumerate(L_values):
    for col, T in enumerate(T_values):
        ax = axes[row, col]
        e  = results[(L, T)]
        N  = np.arange(len(e))

        # dados subamostrados para visualização
        step = max(1, len(e) // 300)
        ax.plot(N[::step], e[::step], lw=0.5, color=f"C{col}", alpha=0.5, label="dados")

        # curva de fit
        ef, tau = fit_params[(L, T)]
        if not np.isnan(ef):
            N_fit = np.linspace(0, N[-1], 500)
            ax.plot(
                N_fit, therm_model(N_fit, ef, tau),
                color="k", lw=1.5, ls="--",
                label=f"fit\n$e_f$={ef:.3f}\n$\\tau$={tau:.0f}"
            )

        ax.set_title(f"$L={L}$,  $T={T}$", fontsize=9)
        ax.legend(fontsize=7, loc="best")
        ax.grid(True, ls="--", alpha=0.4)
        if row == len(L_values) - 1:
            ax.set_xlabel("$N$ (iteração)")
        if col == 0:
            ax.set_ylabel(r"$e = E/L^2$")

plt.tight_layout()
plt.show()

# ── Tabela de resultados ─────────────────────────────────────────────────────
print(f"\n{'L':>5}  {'T':>4}  {'ef':>9}  {'±σ_ef':>9}  {'τ_term':>12}  {'±σ_τ':>12}")
print("─" * 58)
for L in L_values:
    for T in T_values:
        ef,  tau  = fit_params[(L, T)]
        sef, stau = fit_errors [(L, T)]
        print(f"{L:>5}  {T:>4}  {ef:>9.4f}  {sef:>9.4f}  {tau:>12.1f}  {stau:>12.1f}")
